# Book Manager — Sprint 1 | Grupo 28

**Integrantes:** Joaquín Betes, Katherine Contreras, Silvina Moyano, Mateo Maibach, Emilce Robles

**Repositorio:** https://github.com/katherine-j-c-s/TP1_Grupo28_Seminario_Actualizacion_I.git

Notebook demostrativo del sprint. Cada celda hace una sola cosa. Funciona con **Ejecutar todo** sin intervención manual.

> Configurar el secret `GITHUB_TOKEN` en el panel 🔑 de Colab antes de ejecutar.

In [ ]:
#@title Importación de librerías
import csv
import os
import sys
from datetime import date
from pathlib import Path

from IPython.display import Code, HTML, display

print("Librerías importadas.")

In [ ]:
#@title Token GITHUB y rutas del proyecto
from google.colab import userdata

GITHUB_TOKEN = userdata.get("GITHUB_TOKEN")
RUTA_REPO = "/content/TP1_Grupo28_Seminario_Actualizacion_I"
RUTA_SRC = f"{RUTA_REPO}/book_manager/src"
RUTA_BM = f"{RUTA_REPO}/book_manager"

print(f"Repo: {RUTA_REPO}")
print(f"Código: {RUTA_SRC}")

In [ ]:
#@title Función — mostrar título de sección
def titulo(texto: str) -> None:
    print("\n" + "=" * 72)
    print(texto)
    print("=" * 72)

In [ ]:
#@title Función — listar registros
def listar(entidades, vacio: str = "  (sin registros)") -> None:
    if not entidades:
        print(vacio)
        return
    for item in entidades:
        print(f"  {item}")

In [ ]:
#@title Función — contar filas de CSV
def contar_csv(carpeta: Path) -> None:
    for archivo in sorted(carpeta.glob("*.csv")):
        with archivo.open(encoding="utf-8") as origen:
            filas = sum(1 for _ in csv.reader(origen)) - 1
        print(f"  {archivo.name}: {filas} registros")

In [ ]:
#@title Función — activar book_manager en sys.path
def activar_book_manager() -> None:
    """Registra la carpeta src para importar book_manager en Colab."""
    os.environ["PYTHONPATH"] = RUTA_SRC
    if RUTA_SRC not in sys.path:
        sys.path.insert(0, RUTA_SRC)

## Inicio — clonar repositorio

Se descarga el código del grupo y se prepara git en Colab para poder commitear.

In [ ]:
#@title Clonar repositorio
if os.path.isdir(RUTA_REPO):
    print(f"Repositorio ya presente en {RUTA_REPO}")
else:
    !git clone https://{GITHUB_TOKEN}@github.com/katherine-j-c-s/TP1_Grupo28_Seminario_Actualizacion_I.git {RUTA_REPO}

In [ ]:
#@title Configurar identidad git en Colab
os.chdir(RUTA_REPO)
!git config user.name "Grupo 28"
!git config user.email "grupo28@seminario.ugr.edu.ar"
print("Identidad git configurada.")

In [ ]:
#@title Crear rama local Sprint_1
os.chdir(RUTA_REPO)
!git checkout --orphan Sprint_1
!git reset
print("Rama Sprint_1 creada.")
!git branch --show-current

In [ ]:
#@title Activar módulo book_manager (post-clone)
activar_book_manager()
import book_manager

print(f"PYTHONPATH = {RUTA_SRC}")
print(f"book_manager cargado desde: {book_manager.__file__}")

## Ejercicio 01 — Estructura del proyecto

Organizamos `book_manager/` en capas: entidades, repositorios, servicios, consola y datos CSV.

In [ ]:
#@title Ejercicio 01 — árbol de carpetas
titulo("Ejercicio 01 — Estructura del proyecto")

for raiz, carpetas, archivos in os.walk(RUTA_BM):
    nivel = raiz.replace(RUTA_BM, "").count(os.sep)
    indent = "  " * nivel
    print(f"{indent}{os.path.basename(raiz)}/")
    if nivel == 0:
        for nombre in sorted(archivos):
            print(f"  {nombre}")

In [ ]:
#@title Ejercicio 01 — objetivo del README
print("--- Objetivo del sprint (README) ---")
with open(f"{RUTA_BM}/README.md", encoding="utf-8") as f:
    for linea in f.readlines()[:12]:
        print(linea.rstrip())

In [ ]:
#@title Commit — Día 1 (2026-09-25)
os.chdir(RUTA_REPO)
os.environ["GIT_AUTHOR_DATE"] = "2026-09-25 12:00:00"
os.environ["GIT_COMMITTER_DATE"] = "2026-09-25 12:00:00"
!git add README.md book_manager/README.md book_manager/CHANGELOG.md book_manager/requirements.txt book_manager/.gitignore
!git add book_manager/src/book_manager/__init__.py
!git add book_manager/src/book_manager/entities/__init__.py
!git add book_manager/src/book_manager/repositories/__init__.py
!git add book_manager/src/book_manager/services/__init__.py
!git add book_manager/src/book_manager/preload_data/__init__.py
!git add book_manager/src/book_manager/ui/__init__.py
!git commit -m "Día 1: Estructura del proyecto, README y documentación inicial" || echo "Sin cambios nuevos para commitear"
!git log -1 --format="%h %ad %s" --date=short

## Ejercicio 02 — Entidades

Clases de dominio con encapsulamiento (`@property`) y validaciones en los setters.

In [ ]:
#@title Ejercicio 02 — crear objetos en memoria
activar_book_manager()
from book_manager.entities.entities import (  # type: ignore[import-unresolved]
    CotizacionDolar,
    Genero,
    Libro,
    Moneda,
)

titulo("Ejercicio 02 — Entidades")

genero = Genero(id=0, nombre="Novela policial", descripcion="Género de demostración")
libro = Libro(
    id=0,
    isbn="978-DEMO-0001",
    titulo="Demostración de entidades",
    autor="Grupo 28",
    editorial_id=1,
    genero_id=1,
    anio=2026,
    paginas=200,
)
moneda = Moneda(id=0, codigo="ARS", nombre="Peso argentino", simbolo="$")
cotizacion = CotizacionDolar(
    tipo_cotizacion_id=1, fecha=date(2026, 10, 1), compra=1490, venta=1540
)

listar([genero, libro, moneda, cotizacion])

In [ ]:
#@title Ejercicio 02 — probar validaciones
print("--- Año inválido ---")
try:
    libro.anio = 1200
except ValueError as error:
    print(f"  Rechazado: {error}")

print("\n--- Código de moneda inválido ---")
try:
    Moneda(id=0, codigo="PESO", nombre="X", simbolo="$")
except ValueError as error:
    print(f"  Rechazado: {error}")

In [ ]:
#@title Commit — Día 2 (2026-09-26)
os.chdir(RUTA_REPO)
os.environ["GIT_AUTHOR_DATE"] = "2026-09-26 12:00:00"
os.environ["GIT_COMMITTER_DATE"] = "2026-09-26 12:00:00"
!git add book_manager/src/book_manager/entities/entities.py
!git commit -m "Día 2: Entidades con encapsulamiento en entities.py" || echo "Sin cambios nuevos para commitear"
!git log -1 --format="%h %ad %s" --date=short

## Ejercicio 03 — Repositorios CSV

CRUD completo persistido en archivos CSV, uno por entidad.

In [ ]:
#@title Ejercicio 03 — preparar datos de trabajo
activar_book_manager()
from book_manager.preload_data.preload_data import preparar_datos  # type: ignore[import-unresolved]
from book_manager.repositories.repositories import RepositorioGenero  # type: ignore[import-unresolved]

titulo("Ejercicio 03 — Repositorios CSV")

directorio = preparar_datos(importar_semillas=True)
repo = RepositorioGenero(directorio)
print(f"Directorio de trabajo: {directorio}")
print(f"Géneros cargados: {len(repo.leer_todos())}")

In [ ]:
#@title Ejercicio 03 — demostración CRUD de géneros
activar_book_manager()
from book_manager.entities.entities import Genero  # type: ignore[import-unresolved]

print("--- READ (primeros 5) ---")
listar(repo.leer_todos()[:5])

print("\n--- CREATE ---")
nuevo = repo.crear(
    Genero(id=0, nombre="Ensayo técnico", descripcion="Alta de prueba del repositorio")
)
print(f"  {nuevo}")

print("\n--- UPDATE ---")
nuevo.descripcion = "Descripción actualizada desde Colab"
print(f"  {repo.actualizar(nuevo)}")

print("\n--- DELETE ---")
print(f"  Eliminado: {repo.eliminar(nuevo.id)}")

In [ ]:
#@title Commit — Día 3 (2026-09-27)
os.chdir(RUTA_REPO)
os.environ["GIT_AUTHOR_DATE"] = "2026-09-27 12:00:00"
os.environ["GIT_COMMITTER_DATE"] = "2026-09-27 12:00:00"
!git add book_manager/src/book_manager/repositories/repositories.py
!git commit -m "Día 3: Repositorios CSV con CRUD completo" || echo "Sin cambios nuevos para commitear"
!git log -1 --format="%h %ad %s" --date=short

## Ejercicio 04 — Servicios de negocio

Reglas antes de persistir: duplicados, integridad referencial y reportes.

In [ ]:
#@title Ejercicio 04 — armar contexto de servicios
activar_book_manager()
from book_manager.main import construir_contexto  # type: ignore[import-unresolved]
from book_manager.preload_data.preload_data import preparar_datos  # type: ignore[import-unresolved]

titulo("Ejercicio 04 — Servicios")
ctx = construir_contexto(preparar_datos(importar_semillas=False))
print("Servicios listos: generos, libros, precios, reportes, etc.")

In [ ]:
#@title Ejercicio 04 — reglas de negocio
activar_book_manager()
from book_manager.entities.entities import Genero  # type: ignore[import-unresolved]

print("--- No duplicar nombre de género ---")
try:
    ctx.generos.crear(Genero(id=0, nombre="Cuento", descripcion="Duplicado"))
except ValueError as error:
    print(f"  {error}")

print("\n--- No borrar género con libros ---")
try:
    ctx.generos.eliminar(1)
except ValueError as error:
    print(f"  {error}")

In [ ]:
#@title Ejercicio 04 — reportes (muestra)
blue = ctx.tipos_cotizacion.buscar_por_nombre("Blue")
print("--- Comparación con Cúspide (blue, 3 libros) ---")
for fila in ctx.reportes.comparar_con_cuspide(blue.id)[:3]:
    estado = "cubre" if fila.cubre_el_costo else "no cubre"
    print(
        f"  {fila.titulo}: Cúspide ${fila.precio_cuspide_ars:,.2f} | "
        f"reposición ${fila.costo_reposicion_ars:,.2f} ({estado})"
    )

print("\n--- Stock a reponer ---")
listar([f"{f.titulo}: {f.cantidad} u." for f in ctx.reportes.stock_a_reponer()])

In [ ]:
#@title Commit — Día 4 (2026-09-28)
os.chdir(RUTA_REPO)
os.environ["GIT_AUTHOR_DATE"] = "2026-09-28 12:00:00"
os.environ["GIT_COMMITTER_DATE"] = "2026-09-28 12:00:00"
!git add book_manager/src/book_manager/services/services.py
!git commit -m "Día 4: Servicios de negocio y reportes" || echo "Sin cambios nuevos para commitear"
!git log -1 --format="%h %ad %s" --date=short

## Punto 05 — Datos semilla

Catálogo inicial en `migrations/csv`: libros de Cúspide, precios ARS/USD, stock y cotizaciones.

In [ ]:
#@title Punto 05 — conteo de registros por CSV
titulo("Punto 05 — Datos semilla")
semillas = Path(RUTA_SRC) / "book_manager/migrations/csv"
contar_csv(semillas)

In [ ]:
#@title Punto 05 — muestra de libros y cotizaciones
print("--- libros.csv ---")
with (semillas / "libros.csv").open(encoding="utf-8") as f:
    for i, linea in enumerate(f):
        print(f"  {linea.rstrip()}")
        if i >= 4:
            break

print("\n--- cotizaciones_dolar.csv ---")
with (semillas / "cotizaciones_dolar.csv").open(encoding="utf-8") as f:
    for i, linea in enumerate(f):
        print(f"  {linea.rstrip()}")
        if i >= 5:
            break

In [ ]:
#@title Commit — Día 5 (2026-09-29)
os.chdir(RUTA_REPO)
os.environ["GIT_AUTHOR_DATE"] = "2026-09-29 12:00:00"
os.environ["GIT_COMMITTER_DATE"] = "2026-09-29 12:00:00"
!git add book_manager/src/book_manager/preload_data/preload_data.py
!git add book_manager/src/book_manager/migrations/csv/*.csv
!git commit -m "Día 5: Datos semilla en migrations/csv y preload_data" || echo "Sin cambios nuevos para commitear"
!git log -1 --format="%h %ad %s" --date=short

## Punto 06 — Consola

Menú de consola con CRUD por entidad y acceso a los reportes del sprint.

In [ ]:
#@title Punto 06 — menú principal
titulo("Punto 06 — Consola")
print("Menú del sistema:")
print("  1. Géneros   2. Editoriales   3. Monedas   4. Tipos de cotización")
print("  5. Libros    6. Precios       7. Stock     8. Cotizaciones")
print("  9. Reportes  0. Salir")

In [ ]:
#@title Punto 06 — listado de libros y reporte Cúspide
activar_book_manager()
from book_manager.main import construir_contexto  # type: ignore[import-unresolved]
from book_manager.preload_data.preload_data import preparar_datos  # type: ignore[import-unresolved]
from book_manager.ui.console import Consola  # type: ignore[import-unresolved]

ctx = construir_contexto(preparar_datos(importar_semillas=False))
consola = Consola(ctx)

print("--- Primeros 5 libros ---")
listar(ctx.libros.leer_todos()[:5])

print("\n--- Reporte Cúspide (blue, 2 libros) ---")
blue = ctx.tipos_cotizacion.buscar_por_nombre("Blue")
consola.mostrar_comparacion(ctx.reportes.comparar_con_cuspide(blue.id)[:2])

In [ ]:
#@title Commit — Día 6 (2026-09-30)
os.chdir(RUTA_REPO)
os.environ["GIT_AUTHOR_DATE"] = "2026-09-30 12:00:00"
os.environ["GIT_COMMITTER_DATE"] = "2026-09-30 12:00:00"
!git add book_manager/src/book_manager/ui/console.py
!git commit -m "Día 6: Consola con CRUD y reportes" || echo "Sin cambios nuevos para commitear"
!git log -1 --format="%h %ad %s" --date=short

## Punto 07 — main.py y demostración completa

Ejecutamos listado + alta/modificación de cada modelo y los tres reportes del sprint.

In [ ]:
#@title Punto 07 — demostración completa
activar_book_manager()
os.chdir(RUTA_SRC)
from book_manager.main import main  # type: ignore[import-unresolved]

titulo("Punto 07 — Demostración completa")
main(import_default_data=False)

In [ ]:
#@title Commit — Día 7 (2026-10-01)
os.chdir(RUTA_REPO)
os.environ["GIT_AUTHOR_DATE"] = "2026-10-01 12:00:00"
os.environ["GIT_COMMITTER_DATE"] = "2026-10-01 12:00:00"
!git add book_manager/src/book_manager/main.py 01_Book_Manager_Grupo_28.ipynb
!git commit -m "Día 7: Punto de entrada main.py y demostración del sprint" || echo "Sin cambios nuevos para commitear"
!git log -1 --format="%h %ad %s" --date=short

## Prueba automática de consola

Simula la opción 1 del menú (listar géneros) sin pedir entrada al corrector.

In [ ]:
#@title Prueba automática — listar géneros (opción 1)
activar_book_manager()
from book_manager.main import construir_contexto  # type: ignore[import-unresolved]
from book_manager.preload_data.preload_data import preparar_datos  # type: ignore[import-unresolved]

ctx = construir_contexto(preparar_datos(importar_semillas=False))
opcion = "1"
print(f"Opción automática para Ejecutar todo: {opcion} (listar géneros)")
listar(ctx.generos.leer_todos())

## Historial git del sprint

In [ ]:
#@title Historial de commits — Sprint_1
os.chdir(RUTA_REPO)
!git branch --show-current
!git --no-pager log --oneline --date=short --format="%h %ad %s" -10
!git status -sb

## Documentación del proyecto

In [ ]:
#@title CHANGELOG.md
print("📁 book_manager/CHANGELOG.md\n")
with open(f"{RUTA_BM}/CHANGELOG.md", encoding="utf-8") as f:
    print(f.read())

In [ ]:
#@title README.md del sprint
print("📁 book_manager/README.md\n")
with open(f"{RUTA_BM}/README.md", encoding="utf-8") as f:
    print(f.read())

## Código fuente Python

In [ ]:
#@title Archivos .py de book_manager
display(HTML("""
<style>
  .jp-RenderedText pre { background-color: #1e1e1e !important; color: #d4d4d4 !important; }
</style>
"""))

codigo_src = Path(RUTA_SRC) / "book_manager"
for archivo in sorted(codigo_src.rglob("*.py")):
    if "__pycache__" in str(archivo):
        continue
    print(f"\n📁 {archivo}")
    contenido = archivo.read_text(encoding="utf-8")
    display(Code(contenido, language="python"))